# Step 5: Prediction Tool with Gradio Interface
Interactive web interface to predict molecular properties from SMILES

In [1]:
# Install required libraries
!pip install rdkit numpy scikit-learn gradio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 29.1 MB/s eta 0:00:00


In [2]:
# Upload trained models
from google.colab import files
print("Upload: models_svr.pkl (from Step 3)")
uploaded = files.upload()

Upload: models_svr.pkl (from Step 3)


Saving models_svr.pkl to models_svr.pkl


In [3]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
import pickle
import gradio as gr
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("STEP 5: PREDICTION TOOL WITH GRADIO")
print("="*70)

STEP 5: PREDICTION TOOL WITH GRADIO


In [4]:
# Load trained models
print("\nLoading models...")
with open('models_svr.pkl', 'rb') as f:
    models = pickle.load(f)

print("✓ Models loaded successfully")


Loading models...
✓ Models loaded successfully


In [5]:
# Define helper functions

def smiles_to_fingerprint(smiles, radius=2, nBits=2048):
    """Convert SMILES to Morgan fingerprint"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES string")
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nBits)
    return np.array(fp)

def smiles_to_image(smiles):
    """Generate molecular structure image from SMILES"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    img = Draw.MolToImage(mol, size=(400, 400))
    return img

print("✓ Helper functions defined")

✓ Helper functions defined


In [6]:
# Main prediction function for Gradio

def predict_properties(smiles):
    """
    Predict molecular electronic properties from SMILES
    Returns: molecular image, predictions table, and interpretation
    """
    try:
        # Validate and convert SMILES
        fp = smiles_to_fingerprint(smiles)
        fp = fp.reshape(1, -1)

        # Generate molecular image
        mol_image = smiles_to_image(smiles)

        # Make predictions
        homo_pred = models['HOMO'].predict(fp)[0]
        lumo_pred = models['LUMO'].predict(fp)[0]
        eg_pred = models['Eg'].predict(fp)[0]
        calc_eg = lumo_pred - homo_pred

        # Format results as markdown table
        results_table = f"""
### 🎯 Predicted Properties

| Property | Value (eV) | Description |
|----------|------------|-------------|
| **HOMO** | {homo_pred:.3f} | Highest Occupied Molecular Orbital |
| **LUMO** | {lumo_pred:.3f} | Lowest Unoccupied Molecular Orbital |
| **Optical Bandgap** | {eg_pred:.3f} | Measured energy gap |
| **Calculated Bandgap** | {calc_eg:.3f} | LUMO - HOMO |

---

### 📊 Interpretation

**HOMO = {homo_pred:.3f} eV**
- {'Higher' if homo_pred > -5.5 else 'Lower'} than average ({-5.61:.2f} eV)
- {'Easier' if homo_pred > -5.5 else 'Harder'} to oxidize (remove electrons)

**LUMO = {lumo_pred:.3f} eV**
- {'Higher' if lumo_pred > -3.9 else 'Lower'} than average ({-3.93:.2f} eV)
- {'Harder' if lumo_pred > -3.9 else 'Easier'} to reduce (add electrons)

**Optical Bandgap = {eg_pred:.3f} eV**
- {'Wider' if eg_pred > 1.46 else 'Narrower'} than average ({1.46:.2f} eV)
- Absorbs light at ~{1240/eg_pred:.0f} nm wavelength
- {'Visible' if 1.8 < eg_pred < 3.1 else 'IR' if eg_pred < 1.8 else 'UV'} light absorption

**Difference (Optical vs Calculated):** {abs(eg_pred - calc_eg):.3f} eV
- {'Good agreement' if abs(eg_pred - calc_eg) < 0.3 else 'Significant difference'} between optical and calculated bandgap
"""

        return mol_image, results_table

    except Exception as e:
        error_msg = f"""
### ❌ Error

**Could not process SMILES string:** `{smiles}`

**Error message:** {str(e)}

**Please check:**
- SMILES string is valid
- No typos or invalid characters
- Use standard SMILES notation

**Examples of valid SMILES:**
- Benzene: `c1ccccc1`
- Naphthalene: `c1ccc2ccccc2c1`
- Toluene: `Cc1ccccc1`
"""
        return None, error_msg

print("✓ Prediction function defined")

✓ Prediction function defined


In [7]:
# Create Gradio interface

# Example molecules for quick testing
examples = [
    ["c1ccccc1"],  # Benzene
    ["c1ccc2ccccc2c1"],  # Naphthalene
    ["c1ccc2cc3ccccc3cc2c1"],  # Anthracene
    ["Cc1ccccc1"],  # Toluene
    ["c1ccc(cc1)C(=O)O"],  # Benzoic acid
]

# Create the interface
interface = gr.Interface(
    fn=predict_properties,
    inputs=[
        gr.Textbox(
            label="SMILES String",
            placeholder="Enter SMILES (e.g., c1ccccc1 for benzene)",
            lines=1
        )
    ],
    outputs=[
        gr.Image(label="Molecular Structure", type="pil"),
        gr.Markdown(label="Predictions & Analysis")
    ],
    examples=examples,
    title="🧪 Molecular Property Predictor",
    description="""
    Predict **HOMO**, **LUMO**, and **Optical Bandgap** of organic molecules from their SMILES representation.

    This tool uses machine learning models trained on 1,571 acceptor molecules from organic solar cell research.

    **How to use:**
    1. Enter a SMILES string (or click an example below)
    2. Click "Submit" to see predictions
    3. View the molecular structure and predicted properties
    """,
    article="""
    ---

    ### 📖 About

    **Model:** Support Vector Regression (SVR) with RBF kernel

    **Features:** Morgan Fingerprints (radius=2, 2048 bits)

    **Performance:**
    - HOMO: R² = 0.317, MAE = 0.077 eV
    - LUMO: R² = 0.195, MAE = 0.103 eV
    - Optical Bandgap: R² = 0.357, MAE = 0.070 eV

    **Training Data:** 1,571 acceptor molecules from organic photovoltaic research

    **Note:** Predictions are most reliable for molecules similar to the training set (organic semiconductors/acceptors).

    ---

    **Project:** Predicting Optical Bandgaps and Frontier Orbital Energies from SMILES Using Machine Learning
    """,
    theme="soft",
    allow_flagging="never"
)

print("✓ Gradio interface created")

✓ Gradio interface created


In [8]:
# Launch the interface
print("\n" + "="*70)
print("LAUNCHING GRADIO INTERFACE")
print("="*70)
print("\nThe interface will open in a new tab.")
print("You can share the public URL with your team!\n")

interface.launch(share=True, debug=False)


LAUNCHING GRADIO INTERFACE

The interface will open in a new tab.
You can share the public URL with your team!

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://73ce7b65ae55f4f1e7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

## 🎉 Your Gradio App is Now Running!

### How to Use:

1. **Click the link above** to open the interface
2. **Enter a SMILES string** or click one of the examples
3. **Click Submit** to get predictions
4. **Share the public URL** with your team members!

### Features:

✅ **Interactive Web Interface** - No coding required for users

✅ **Molecular Structure Visualization** - See the molecule you're predicting

✅ **Detailed Predictions** - HOMO, LUMO, and Bandgap with interpretations

✅ **Example Molecules** - Quick testing with pre-loaded examples

✅ **Shareable Link** - Anyone can access via the public URL

✅ **Error Handling** - Clear messages for invalid inputs

### Tips:

- The public link will be active as long as this notebook is running
- You can share the link with your team/instructor
- Test with the example molecules first
- Valid SMILES: `c1ccccc1` (benzene), `CC(=O)O` (acetic acid)

---

In [9]:
# Optional: Batch prediction with CSV upload

def batch_predict_from_csv(csv_file):
    """
    Process multiple SMILES from uploaded CSV file
    CSV should have a column named 'SMILES'
    """
    import pandas as pd

    try:
        # Read CSV
        df = pd.read_csv(csv_file.name)

        if 'SMILES' not in df.columns:
            return "Error: CSV must have a 'SMILES' column"

        # Predict for each SMILES
        results = []
        for smiles in df['SMILES']:
            try:
                fp = smiles_to_fingerprint(smiles)
                fp = fp.reshape(1, -1)

                homo = models['HOMO'].predict(fp)[0]
                lumo = models['LUMO'].predict(fp)[0]
                eg = models['Eg'].predict(fp)[0]

                results.append({
                    'SMILES': smiles,
                    'HOMO_pred': round(homo, 3),
                    'LUMO_pred': round(lumo, 3),
                    'Eg_opt_pred': round(eg, 3),
                    'Calc_Eg': round(lumo - homo, 3)
                })
            except:
                results.append({
                    'SMILES': smiles,
                    'HOMO_pred': 'Error',
                    'LUMO_pred': 'Error',
                    'Eg_opt_pred': 'Error',
                    'Calc_Eg': 'Error'
                })

        # Create results DataFrame
        results_df = pd.DataFrame(results)

        # Save to CSV
        output_file = 'predictions_output.csv'
        results_df.to_csv(output_file, index=False)

        return results_df, output_file

    except Exception as e:
        return f"Error processing CSV: {str(e)}", None

# Create batch prediction interface
batch_interface = gr.Interface(
    fn=batch_predict_from_csv,
    inputs=gr.File(label="Upload CSV with SMILES column"),
    outputs=[
        gr.Dataframe(label="Predictions"),
        gr.File(label="Download Results")
    ],
    title="📊 Batch Prediction (CSV Upload)",
    description="Upload a CSV file with a 'SMILES' column to predict properties for multiple molecules at once."
)

print("\n✓ Batch prediction interface ready")
print("\nUncomment the line below to launch batch prediction:")
# batch_interface.launch(share=True)


✓ Batch prediction interface ready

Uncomment the line below to launch batch prediction:


In [10]:
# Save standalone prediction script (for local use)

standalone_script = '''#!/usr/bin/env python3
"""
Molecular Property Prediction Tool - Gradio Version
Run locally: python predict_gradio.py
"""

import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
import pickle
import gradio as gr
import warnings
warnings.filterwarnings('ignore')

# Load models
with open('models_svr.pkl', 'rb') as f:
    models = pickle.load(f)

def smiles_to_fingerprint(smiles, radius=2, nBits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES")
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nBits)
    return np.array(fp)

def smiles_to_image(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Draw.MolToImage(mol, size=(400, 400))

def predict_properties(smiles):
    try:
        fp = smiles_to_fingerprint(smiles).reshape(1, -1)
        mol_image = smiles_to_image(smiles)

        homo = models['HOMO'].predict(fp)[0]
        lumo = models['LUMO'].predict(fp)[0]
        eg = models['Eg'].predict(fp)[0]

        results = f"""### Predictions\n\n
- **HOMO:** {homo:.3f} eV\n
- **LUMO:** {lumo:.3f} eV\n
- **Optical Bandgap:** {eg:.3f} eV\n
- **Calculated Bandgap:** {lumo-homo:.3f} eV"""

        return mol_image, results
    except Exception as e:
        return None, f"Error: {str(e)}"

# Create interface
interface = gr.Interface(
    fn=predict_properties,
    inputs=gr.Textbox(label="SMILES", placeholder="c1ccccc1"),
    outputs=[gr.Image(label="Molecule"), gr.Markdown(label="Predictions")],
    examples=[["c1ccccc1"], ["c1ccc2ccccc2c1"]],
    title="Molecular Property Predictor"
)

if __name__ == "__main__":
    interface.launch(share=True)
'''

with open('predict_gradio.py', 'w') as f:
    f.write(standalone_script)

print("\n✓ Saved: predict_gradio.py")
print("\nFor local use: python predict_gradio.py")


✓ Saved: predict_gradio.py

For local use: python predict_gradio.py


In [11]:
# Download the standalone script
from google.colab import files
files.download('predict_gradio.py')

print("\n" + "="*70)
print("✓ STEP 5 COMPLETE - Gradio Interface Ready!")
print("="*70)
print("\nYour prediction tool is now running with a user-friendly interface.")
print("Share the public link with your team and instructor!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ STEP 5 COMPLETE - Gradio Interface Ready!

Your prediction tool is now running with a user-friendly interface.
Share the public link with your team and instructor!
